# Stage 8 Discrete N-Fold Optical Fields

This notebook treats finite N-fold beams as discrete plane-wave
superpositions on one transverse-k ring. These are optical fields and
geometry diagnostics only. A clean N-fold transverse pattern is not a
material-writing result, and a focal or short-z symmetry metric is not
a stable written channel.

Ideal N-wave rows are simulation-only or future-hardware targets.
Phase-only proxy rows are labelled current-lab-realizable only for the
encoded phase-only optical command, not for material modification.


## Stage 8.7 Adjustable Quick-Look Guidance

<!-- STAGE87: adjustable quicklook guidance -->

For fast parameter scouting, use `notebooks/quicklook/00_quick_beam_to_sample_simulator.ipynb`. This notebook remains on its locked stage path: existing execution logic, propagation-power labels, material-proxy caveats, and governance routing are unchanged.

Safe local edits are the explicit config variables already exposed by this notebook, or a copied exploratory run. Keep `fail` and `marginal` labels visible. If a displayed image is visually smoothed, treat that as display interpolation only; rerun balanced/publication sampling before numerical interpretation.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import bessel_twin_core as bt
from vbb_study import setup_study
from vbb_study.equations import polygonal
from vbb_study.publication import advanced as advanced_schema
from vbb_study.studies import discrete_nfold_cases

PATHS = setup_study.bootstrap(Path.cwd())
RUN_ID = PATHS.get("run_id") or None
CSV_OUT = PATHS["csv"] / "advanced"
CSV_OUT.mkdir(parents=True, exist_ok=True)
(PATHS["csv"] / "stage10_discrete").mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 120)


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [ ]:
# STAGE88: visible editable controls for exploratory notebook use.
from vbb_study.publication import notebook_controls as nb_controls

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(stage='advanced')
try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


## Field And Metric Helpers

The ideal branch uses a finite N-wave complex field. The phase-only
branch keeps only the phase of that target under a Gaussian envelope.
Both branches are propagated numerically and scored with symmetry,
side-lobe, core, and accepted-depth metrics.


In [2]:
grid = bt.make_xy_grid(128, 0.35 * bt.um)
R = np.asarray(grid["R"], dtype=float)
PHI = np.asarray(grid["PHI"], dtype=float)
kr_m_inv = 1.55 / bt.um
waist_m = 28.0 * bt.um
wavelength_m = 1030.0 * bt.nm
z_values_m = np.linspace(0.0, 70.0 * bt.um, 8)
gaussian = bt.gaussian_amplitude(R, waist_m)

def angular_order_metrics(intensity, grid, expected_order):
    I = np.asarray(intensity, dtype=float)
    Rg = np.asarray(grid["R"], dtype=float)
    PHIg = np.asarray(grid["PHI"], dtype=float) % (2.0 * np.pi)
    r0 = 2.0 * np.pi / kr_m_inv
    annulus = (Rg >= 0.45 * r0) & (Rg <= 2.6 * r0)
    bins = np.linspace(0.0, 2.0 * np.pi, 361)
    profile = np.zeros(360, dtype=float)
    counts = np.zeros(360, dtype=float)
    idx = np.clip(np.digitize(PHIg[annulus], bins) - 1, 0, 359)
    np.add.at(profile, idx, I[annulus])
    np.add.at(counts, idx, 1.0)
    filled = counts > 0
    profile[filled] /= counts[filled]
    amps = np.abs(np.fft.rfft(profile - float(np.mean(profile))))
    if amps.size:
        amps[0] = 0.0
    measured = int(np.argmax(amps)) if amps.size else 0
    expected_amp = float(amps[int(expected_order)]) if int(expected_order) < amps.size else 0.0
    other = np.array(amps, copy=True)
    if int(expected_order) < other.size:
        other[int(expected_order)] = 0.0
    score = expected_amp / (expected_amp + float(np.max(other)) + bt.EPS)
    return measured, float(np.clip(score, 0.0, 1.0))

def score_plane(plane, grid, expected_order):
    I = np.asarray(plane, dtype=float)
    norm = I / (float(np.max(I)) + bt.EPS)
    Rg = np.asarray(grid["R"], dtype=float)
    r0 = 2.0 * np.pi / kr_m_inv
    signal = (Rg >= 0.45 * r0) & (Rg <= 2.6 * r0)
    eval_mask = Rg <= 6.0 * r0
    measured, symmetry = angular_order_metrics(norm, grid, expected_order)
    return {
        "measured_symmetry_order": measured,
        "symmetry_score": symmetry,
        "outline_fidelity_score": pd.NA,
        "edge_uniformity_score": polygonal.edge_uniformity_score(norm, signal),
        "core_suppression_score": polygonal.core_suppression_score(
            norm,
            Rg,
            core_radius_m=0.45 * r0,
            reference_mask=signal,
        ),
        "side_lobe_contamination_score": polygonal.side_lobe_contamination_score(
            norm,
            signal,
            evaluation_mask=eval_mask,
        ),
    }


## N-Fold Propagation Suite

The accepted depth is the longest contiguous z range where the measured
angular order matches the requested order and the symmetry score stays
above the explicit threshold. Passing this gate is an optical
propagation statement only.


In [3]:
rows = []
z_detail_rows = []
for case in discrete_nfold_cases.discrete_nfold_stage8_cases():
    order = int(case["target_symmetry_order"])
    ell = int(case.get("ell", 0))
    target_field = polygonal.discrete_nfold_field(
        R,
        PHI,
        kr_m_inv=kr_m_inv,
        N=order,
        ell=ell,
    ) * gaussian
    if case["path"] == "phase_only_proxy":
        U0 = np.exp(1j * np.angle(target_field)) * gaussian
        hardware_status = "current_lab_realizable"
    else:
        U0 = target_field
        hardware_status = "simulation_only"
    volume = bt.propagate_volume(
        U0,
        grid,
        wavelength_m,
        z_values_m,
        n_medium=1.0,
        crop_pixels=128,
        bandlimit=True,
        method="bl_asm",
    )
    accepted = []
    per_z = []
    for idx, z_m in enumerate(z_values_m):
        metrics = score_plane(volume["intensity_stack"][idx], volume["crop_grid"], order)
        pass_plane = bool(
            int(metrics["measured_symmetry_order"]) == order
            and float(metrics["symmetry_score"]) >= 0.18
        )
        accepted.append(pass_plane)
        per_z.append({**metrics, "z_um": float(z_m / bt.um), "accepted": pass_plane})
    depth = polygonal.accepted_depth_from_metric_stack(z_values_m, accepted)
    summary_metrics = {
        "measured_symmetry_order": int(round(pd.Series([p["measured_symmetry_order"] for p in per_z]).mode().iloc[0])),
        "symmetry_score": float(np.mean([p["symmetry_score"] for p in per_z])),
        "edge_uniformity_score": float(np.mean([p["edge_uniformity_score"] for p in per_z])),
        "core_suppression_score": float(np.mean([p["core_suppression_score"] for p in per_z])),
        "side_lobe_contamination_score": float(np.mean([p["side_lobe_contamination_score"] for p in per_z])),
        "accepted_depth_um": depth["accepted_depth_um"],
        "accepted_depth_fraction": depth["accepted_depth_fraction"],
        "accepted_plane_count": depth["accepted_plane_count"],
        "accepted_z_start_um": depth["accepted_z_start_um"],
        "accepted_z_end_um": depth["accepted_z_end_um"],
        "accepted_depth_definition": "longest contiguous z span with measured order N and symmetry_score >= 0.18",
        "canonical_zone_um": float(depth["accepted_depth_um"]),
        "strict_bessel_region_um": float(depth["accepted_depth_um"]) if depth["accepted_depth_fraction"] >= 0.65 else 0.0,
        "propagation_power_drift_fraction": float(
            (np.max(volume["total_power"]) - np.min(volume["total_power"]))
            / (np.mean(volume["total_power"]) + bt.EPS)
        ),
    }
    row = advanced_schema.annotate_advanced_beam_row({
        **case,
        **summary_metrics,
        "hardware_status": hardware_status,
        "optical_model_status": "numerical_propagation",
        "material_model_status": "optical_only",
        "calibration_status": "uncalibrated",
    }, run_id=RUN_ID)
    rows.append(row)
    for idx, detail in enumerate(per_z):
        z_detail_rows.append(advanced_schema.annotate_advanced_beam_row({
            **case,
            **detail,
            "case_id": f"{case['case_id']}_z{idx:02d}",
            "hardware_status": hardware_status,
            "optical_model_status": "numerical_propagation",
            "material_model_status": "optical_only",
            "calibration_status": "uncalibrated",
            "accepted_depth_um": depth["accepted_depth_um"],
            "accepted_depth_fraction": depth["accepted_depth_fraction"],
            "accepted_depth_definition": "parent case accepted-depth gate",
            "propagation_power_drift_fraction": summary_metrics["propagation_power_drift_fraction"],
        }, run_id=RUN_ID))

summary = advanced_schema.ordered_advanced_beam_frame(rows)
z_profile = advanced_schema.ordered_advanced_beam_frame(z_detail_rows)
display(summary[[
    "case_id",
    "target_symmetry_order",
    "path",
    "generation_method",
    "hardware_status",
    "symmetry_score",
    "accepted_depth_um",
    "propagation_stability_status",
    "phase_only_compatible",
    "complex_amplitude_required",
]])


,case_id,target_symmetry_order,path,generation_method,hardware_status,symmetry_score,accepted_depth_um,propagation_stability_status,phase_only_compatible,complex_amplitude_required
0,triangular_ideal_discrete_superposition,3,ideal,discrete_superposition,simulation_only,0.027927,0.0,propagation_tested_fail,False,True
1,triangular_phase_only_approximation,3,phase_only_proxy,phase_only_slm,current_lab_realizable,0.084036,0.0,propagation_tested_fail,True,False
2,square_ideal_discrete_superposition,4,ideal,discrete_superposition,simulation_only,0.035089,0.0,propagation_tested_fail,False,True
3,square_phase_only_approximation,4,phase_only_proxy,phase_only_slm,current_lab_realizable,0.111544,0.0,propagation_tested_fail,True,False
4,hexagonal_ideal_discrete_superposition,6,ideal,discrete_superposition,simulation_only,0.283787,0.0,propagation_tested_fail,False,True
5,hexagonal_phase_only_approximation,6,phase_only_proxy,phase_only_slm,current_lab_realizable,0.233774,0.0,propagation_tested_fail,True,False
6,octagonal_ideal_discrete_superposition,8,ideal,discrete_superposition,simulation_only,0.552813,70.0,propagation_tested_pass,False,True
7,octagonal_phase_only_approximation,8,phase_only_proxy,phase_only_slm,current_lab_realizable,0.486984,50.0,propagation_tested_pass,True,False
8,dodecagonal_ideal_discrete_superposition,12,ideal,discrete_superposition,simulation_only,0.319498,0.0,propagation_tested_fail,False,True
9,dodecagonal_phase_only_approximation,12,phase_only_proxy,phase_only_slm,current_lab_realizable,0.297931,0.0,propagation_tested_fail,True,False


## Canonical Outputs

Canonical Stage 8 discrete CSVs are written under
`outputs/csv/advanced`. Older `stage10_discrete` names are refreshed as
schema-native compatibility copies. The old material-threshold view is
intentionally not regenerated as a material claim.


In [4]:
acceptance = summary.copy()
acceptance["acceptance_check"] = acceptance["advanced_acceptance_label"]
acceptance["acceptance_pass"] = acceptance["propagation_stability_status"].isin([
    "propagation_tested_pass",
    "propagation_tested_marginal",
])
summary_path = CSV_OUT / "discrete_nfold_beam_summary.csv"
acceptance_path = CSV_OUT / "discrete_nfold_acceptance_summary.csv"
summary.to_csv(summary_path, index=False)
acceptance.to_csv(acceptance_path, index=False)

compat_dir = PATHS["csv"] / "stage10_discrete"
for name in (
    "10_discrete_pattern_summary.csv",
    "10_discrete_cgh_exports.csv",
    "10_discrete_encoding_comparison.csv",
    "10_discrete_continuous_limit.csv",
):
    summary.to_csv(compat_dir / name, index=False)
acceptance.to_csv(compat_dir / "10_discrete_acceptance_summary.csv", index=False)

assert not summary["material_writing_success_claimed"].astype(bool).any()
assert not summary["stable_written_channel_claimed"].astype(bool).any()
assert not (summary["simulation_only"].astype(bool) & summary["current_lab_realizable"].astype(bool)).any()
assert not (
    summary["complex_amplitude_required"].astype(bool)
    & summary["phase_only_compatible"].astype(bool)
).any()
display(acceptance[[
    "case_id",
    "acceptance_pass",
    "advanced_acceptance_label",
    "hardware_status",
    "propagation_stability_status",
    "material_writing_success_claimed",
    "stable_written_channel_claimed",
]])
print(summary_path)
print(acceptance_path)


,case_id,acceptance_pass,advanced_acceptance_label,hardware_status,propagation_stability_status,material_writing_success_claimed,stable_written_channel_claimed
0,triangular_ideal_discrete_superposition,False,propagation_tested_fail,simulation_only,propagation_tested_fail,False,False
1,triangular_phase_only_approximation,False,propagation_tested_fail,current_lab_realizable,propagation_tested_fail,False,False
2,square_ideal_discrete_superposition,False,propagation_tested_fail,simulation_only,propagation_tested_fail,False,False
3,square_phase_only_approximation,False,propagation_tested_fail,current_lab_realizable,propagation_tested_fail,False,False
4,hexagonal_ideal_discrete_superposition,False,propagation_tested_fail,simulation_only,propagation_tested_fail,False,False
5,hexagonal_phase_only_approximation,False,propagation_tested_fail,current_lab_realizable,propagation_tested_fail,False,False
6,octagonal_ideal_discrete_superposition,True,propagation_tested_simulation_candidate,simulation_only,propagation_tested_pass,False,False
7,octagonal_phase_only_approximation,True,propagation_tested_current_lab_candidate,current_lab_realizable,propagation_tested_pass,False,False
8,dodecagonal_ideal_discrete_superposition,False,propagation_tested_fail,simulation_only,propagation_tested_fail,False,False
9,dodecagonal_phase_only_approximation,False,propagation_tested_fail,current_lab_realizable,propagation_tested_fail,False,False


C:\PhD\Code\Publication_Study\outputs\csv\advanced\discrete_nfold_beam_summary.csv
C:\PhD\Code\Publication_Study\outputs\csv\advanced\discrete_nfold_acceptance_summary.csv
